In [2]:
import pandas as pd
import datetime
from sklearn import datasets
import logging


In [3]:
from evidently.ui.workspace.cloud import CloudWorkspace

from evidently.report import Report
from evidently.metric_preset import DataQualityPreset
from evidently.metric_preset import DataDriftPreset
from evidently.metrics import *
from evidently.test_suite import TestSuite
from evidently.tests import *
from evidently.test_preset import DataDriftTestPreset
from evidently.tests.base_test import TestResult, TestStatus
from src.DataOps.feature_engg import feature_engg_class
from src.ModelOps.model_fit import ModelFit
import wandb

/Users/sakshamgulati/.local/share/virtualenvs/MLOps_Template-k_6XISXV/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
from dotenv import load_dotenv
import os

load_dotenv()
#connecting to evidently cloud workspace using API token
ws = CloudWorkspace(
token=os.getenv('evi_api'),
url="https://app.evidently.cloud")

# #creating a project in the workspace
__project_name__="Test2 Project"
__team_id__="74fdd884-9679-4693-819b-a6695e652e25"

project = ws.create_project(__project_name__,team_id=__team_id__)
project.description = "My project description"
project.save()

Project(id=UUID('76ed8004-a44e-4fed-9078-5116ef5869fe'), name='Test2 Project', description='My project description', dashboard=DashboardConfig(name='Test2 Project', panels=[], tabs=[], tab_id_to_panel_ids={}), team_id=UUID('74fdd884-9679-4693-819b-a6695e652e25'), date_from=None, date_to=None, created_at=datetime.datetime(2024, 7, 4, 14, 43, 20, 882933))

In [5]:
print(project.id)

76ed8004-a44e-4fed-9078-5116ef5869fe


In [6]:
stock_data = feature_engg_class()
data=stock_data.request_stock_price_hist('AAPL')
stock_data.data=data
train,test=stock_data.split(use_prophet=True)

INFO:root:Feature Engineering class initialized
INFO:root:Retrieving stock price data from Alpha Vantage (This may take a while)...
INFO:root:Data has been successfully downloaded...
INFO:root:Storing the retrieved data into a dataframe...
100%|██████████| 1287/1287 [00:01<00:00, 764.30it/s]
INFO:root:Sorting the index and changing column data types...
INFO:root:Data loaded successfully
INFO:root:Data shape: (1287, 7)
INFO:root:Earliest Date:2024-07-03 00:00:00,Latest Date:2024-07-03 00:00:00
INFO:root:Data prepared for Prophet
INFO:root:Data split into train and test successfully
INFO:root:Train shape: 900
INFO:root:Test shape: 387


In [7]:
import yaml
with open('conf/mlops.yaml', 'r') as file:
            config = yaml.safe_load(file)
model_name = config['model_name']
project_name=config['project_name']
model_type=config['model_type']
wandb_entity=config['wandb_entity']
print(model_name,project_name,model_type,wandb_entity)

stock_price_prediction ml-ops-template time-series sakshamgulati123


In [8]:
from prophet.serialize import  model_from_json

run = wandb.init(project=model_name,job_type=model_type)
artifact = run.use_artifact(f'{wandb_entity}/model-registry/{model_name}:latest', type='model')
artifact_dir = artifact.download()
logging.info(f"Artifact downloaded at: {artifact_dir}")
logging.info("Model artifact downloaded")
#load  pickle file
file_path = os.path.join(artifact_dir, 'serialized_model.json')  # specify the correct file path
with open(file_path, 'r') as fin:
    model = model_from_json(fin.read())  # Load model
logging.info("Model loaded from the registry")
y_pred = model.predict(test)


ERROR:wandb.jupyter:Failed to detect the name of this notebook, you can set it manually with the WANDB_NOTEBOOK_NAME environment variable to enable code saving.
wandb: Currently logged in as: sakshamgulati123. Use `wandb login --relogin` to force relogin


wandb:   1 of 1 files downloaded.  
INFO:root:Artifact downloaded at: /Users/sakshamgulati/MLOps_Template/artifacts/run-77xhl9ch-serialized_model.json:v0
INFO:root:Model artifact downloaded
INFO:root:Model loaded from the registry


In [9]:
y_pred

,ds,trend,yhat_lower,yhat_upper,trend_lower,trend_upper,Christmas Day,Christmas Day_lower,Christmas Day_upper,Christmas Day (observed),...,weekly,weekly_lower,weekly_upper,yearly,yearly_lower,yearly_upper,multiplicative_terms,multiplicative_terms_lower,multiplicative_terms_upper,yhat
0,2017-02-10,-2.444415,-92.465630,67.474760,-2.444415,-2.444415,0.0,0.0,0.0,0.0,...,-7.539187,-7.539187,-7.539187,-0.982402,-0.982402,-0.982402,0.0,0.0,0.0,-10.966004
1,2017-02-17,-4.865519,-93.381155,66.154373,-4.865519,-4.865519,0.0,0.0,0.0,0.0,...,-7.539187,-7.539187,-7.539187,0.392142,0.392142,0.392142,0.0,0.0,0.0,-12.012564
2,2017-02-24,-7.286623,-97.568113,65.067974,-7.286623,-7.286623,0.0,0.0,0.0,0.0,...,-7.539187,-7.539187,-7.539187,-0.541140,-0.541140,-0.541140,0.0,0.0,0.0,-15.366950
3,2017-03-03,-9.707727,-105.995506,62.983282,-9.707727,-9.707727,0.0,0.0,0.0,0.0,...,-7.539187,-7.539187,-7.539187,-1.260896,-1.260896,-1.260896,0.0,0.0,0.0,-18.507810
4,2017-03-10,-12.128831,-96.951751,62.862446,-12.128831,-12.128831,0.0,0.0,0.0,0.0,...,-7.539187,-7.539187,-7.539187,0.968726,0.968726,0.968726,0.0,0.0,0.0,-18.699292
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
382,2024-06-07,-927.306164,-1273.859499,-533.043976,-1262.221650,-526.120102,0.0,0.0,0.0,0.0,...,-7.539187,-7.539187,-7.539187,4.166532,4.166532,4.166532,0.0,0.0,0.0,-930.678819
383,2024-06-14,-929.727268,-1291.924321,-533.886814,-1266.524928,-526.692469,0.0,0.0,0.0,0.0,...,-7.539187,-7.539187,-7.539187,-10.543348,-10.543348,-10.543348,0.0,0.0,0.0,-947.809803
384,2024-06-21,-932.148372,-1316.235273,-551.482662,-1270.777958,-527.264836,0.0,0.0,0.0,0.0,...,-7.539187,-7.539187,-7.539187,-21.683830,-21.683830,-21.683830,0.0,0.0,0.0,-961.371389
385,2024-06-28,-934.569476,-1325.464322,-557.923503,-1274.323628,-527.837203,0.0,0.0,0.0,0.0,...,-7.539187,-7.539187,-7.539187,-23.555259,-23.555259,-23.555259,0.0,0.0,0.0,-965.663922


In [10]:
ref_dataset = run.use_artifact(f'{wandb_entity}/{model_name}/reference-dataset:latest', type='dataset')
ref_dataset_dir = ref_dataset.download()
logging.info(f"Artifact downloaded at: {ref_dataset_dir}")
logging.info("Reference artifact downloaded")

wandb:   1 of 1 files downloaded.  
INFO:root:Artifact downloaded at: /Users/sakshamgulati/MLOps_Template/artifacts/reference-dataset:v5
INFO:root:Reference artifact downloaded


In [11]:
import glob
csv_files = glob.glob(os.path.join(ref_dataset_dir, "*.csv"))
for csv in csv_files:
    reference_data=pd.read_csv(csv)


In [12]:
test

,ds,y
900,2017-02-10,132.12
901,2017-02-17,135.72
902,2017-02-24,136.66
903,2017-03-03,139.78
904,2017-03-10,139.14
...,...,...
1282,2024-06-07,196.89
1283,2024-06-14,212.49
1284,2024-06-21,207.49
1285,2024-06-28,210.62


In [13]:
from evidently.pipeline.column_mapping import ColumnMapping
from evidently.report import Report
from evidently.metric_preset import DataDriftPreset, RegressionPreset

target = 'y'
prediction = 'prediction'
# numerical_features = ['open', 'high', 'low', 'adjusted close', 'volume', 'dividend amount']
# categorical_features = ['season', 'holiday', 'workingday', ]#'weathersit']

column_mapping = ColumnMapping()

column_mapping.target = target
column_mapping.prediction = prediction
# column_mapping.datetime_features = 'ds'
column_mapping.datetime='ds'
column_mapping.id = None
                          

In [14]:
regression_performance_report = Report(metrics=[
    RegressionPreset(),
])

regression_performance_report.run(reference_data=reference_data, current_data=test,
                                        column_mapping=column_mapping)

In [15]:
with open('conf/mlops.yaml', 'r') as file:
        config = yaml.safe_load(file)
        model_name = config['model_name']
        project_name=config['project_name']
        model_type=config['model_type']
        wandb_entity=config['wandb_entity']

In [16]:
project_name

'ml-ops-template'

In [17]:
projects=ws.search_project(project_name)
projects[0].id


UUID('19726c56-b662-4013-93c8-5fb302e2e263')

wandb: Network error (ConnectionError), entering retry loop.
